# Линейная регресия
Для реализации линейной регрессии будет использоваться библиотека pytorch, я не буду писать руками функцию матричного умножения и некоторые другие аспекты, однако постараюсь расписать все аспекты максимально явно, без использования  встроенных функций

Линейная регрессия (Linear regression) — один из простейший алгоритмов машинного обучения, описывающий зависимость целевой переменной от признака в виде линейной функции $y = kx + b$. В данном случае была представлена простая или парная линейная регрессия, а уравнение вида $f_{w, b}(x) = w_0x_0 + w_1x_1 +... + w_{n}x_{n} + b = w \cdot x + b$ называется множественной линейной регрессией, где b — смещение модели, w — вектор её весов, а x — вектор признаков одного обучающего образца.

К другим условиям определения линейной регрессии относятся гомоскедастичность (дисперсия остатков постоянна и конечна), а также отсутствие мильтиколлинеарности (линейной зависимости между признаками).

В качестве функции ошибки мы будем использовать метод наименьших квадратов

Выбор регрессионной линии (плоскости), описывающей взаимосвязь данных наилучшим образом, заключается в минимизации функции потерь J, представленной в виде среднеквадратичной ошибки. Проще говоря, линия должна проходить через данные таким образом, чтобы в среднем разница квадратов ожидаемых и реальных значений была минимальна. Данный метод называется методом наименьших квадратов.

При наличии же большого количества выбросов в данных, более эффективным может оказаться метод наименьших модулей, однако у него есть один серьёзный недостаток: функция модуля не является дифференцируемой в точке x=0, что в ряде случаев может затруднить минимизацию ошибки модели. Следовательно, исходя из теоремы Гаусса-Маркова, метод наименьших квадратов является наиболее оптимальной оценкой параметров модели среди всех линейных и несмещённых оценок за счёт меньшей дисперсии.

Для метрики качества возьмем среднеквадратичную ошибку $mse = \frac{1}{n} \sum_0^n((y_{true} - y_{pred})^2)$ 

## Импорты

In [73]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

Теперь напишем класс для нашей линейной регрессии

In [74]:
class LinearRegression():
    
    # функция инициализации
    def __init__(self, feature_number):
        self.weights = torch.rand(feature_number, requires_grad=True)
        self.bias = torch.rand(1, requires_grad=True)
        self.loss_number = None

    # функция прохода вперед
    def forward(self, data):
        return data @ self.weights + self.bias
    
    # функция ощибки
    def loss(self, true_data, preds): 
        return torch.mean((true_data - preds)**2)

    # функция обучения
    def fit(self, train_data, test_data, epochs=100, learning_rate=0.001):
        train_data = torch.tensor(train_data.values, dtype=torch.float32)
        test_data = torch.tensor(test_data.values, dtype=torch.float32)
        for epoch in range(epochs):
            preds = self.forward(train_data)
            loss = self.loss(test_data, preds) #записываем ошибку в переменную loss
            loss.backward() # считаем градиенты loss относительно self.weights и self.bias

            with torch.no_grad():
                # обновляем веса
                self.weights -= learning_rate * self.weights.grad
                self.bias -= learning_rate * self.bias.grad

                # зануляем градиенты
                self.weights.grad.zero_()
                self.bias.grad.zero_()
                
                self.loss_number = loss.item()

                if epoch % 100 == 0:
                    print(f'Epoch: {epoch}')
                    print(f'loss: {self.loss_number}')

    # функция предсказания
    def predict(self, data):
        data = torch.tensor(data.values, dtype=torch.float32)
        data = torch.as_tensor(data, dtype=torch.float32)

        with torch.no_grad():
            return self.forward(data)

Мы написали класс, но нужно куда-ниибудь применить модель, возьмем какой-нибудь простенький датасет с численными фичами, чтобы не заниматься сейчас feature инжинерингом. 
Я выбрал датасет https://www.kaggle.com/competitions/home-data-for-ml-course/data, где мы предсказываем стоимость дома, скачаем его и решим

In [75]:
train_data = pd.read_csv('datasets/house_pricing/train.csv', index_col='Id')
test_data = pd.read_csv('datasets/house_pricing/test.csv', index_col='Id')

train_data.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
Id,,,,,,,,,,,,,,,,,,,,,
1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


Поскольку мне лень сейчас заниматся фичами, просто дропнем все категориальные фичи и строки с NaN для train, а для теста заполним все NaN'ы медианным значением

In [76]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 1 to 1460
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   str    
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   str    
 5   Alley          91 non-null     str    
 6   LotShape       1460 non-null   str    
 7   LandContour    1460 non-null   str    
 8   Utilities      1460 non-null   str    
 9   LotConfig      1460 non-null   str    
 10  LandSlope      1460 non-null   str    
 11  Neighborhood   1460 non-null   str    
 12  Condition1     1460 non-null   str    
 13  Condition2     1460 non-null   str    
 14  BldgType       1460 non-null   str    
 15  HouseStyle     1460 non-null   str    
 16  OverallQual    1460 non-null   int64  
 17  OverallCond    1460 non-null   int64  
 18  YearBuilt      1460

In [77]:
train_data = train_data.select_dtypes(include='number')
train_data.head()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
Id,,,,,,,,,,,,,,,,,,,,,
1,60,65.0,8450,7,5,2003,2003,196.0,706,0,...,0,61,0,0,0,0,0,2,2008,208500
2,20,80.0,9600,6,8,1976,1976,0.0,978,0,...,298,0,0,0,0,0,0,5,2007,181500
3,60,68.0,11250,7,5,2001,2002,162.0,486,0,...,0,42,0,0,0,0,0,9,2008,223500
4,70,60.0,9550,7,5,1915,1970,0.0,216,0,...,0,35,272,0,0,0,0,2,2006,140000
5,60,84.0,14260,8,5,2000,2000,350.0,655,0,...,192,84,0,0,0,0,0,12,2008,250000


In [78]:
train, val = train_test_split(train_data, random_state=42, test_size=0.1)
train.info()

<class 'pandas.DataFrame'>
Index: 1314 entries, 908 to 1127
Data columns (total 37 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1314 non-null   int64  
 1   LotFrontage    1077 non-null   float64
 2   LotArea        1314 non-null   int64  
 3   OverallQual    1314 non-null   int64  
 4   OverallCond    1314 non-null   int64  
 5   YearBuilt      1314 non-null   int64  
 6   YearRemodAdd   1314 non-null   int64  
 7   MasVnrArea     1306 non-null   float64
 8   BsmtFinSF1     1314 non-null   int64  
 9   BsmtFinSF2     1314 non-null   int64  
 10  BsmtUnfSF      1314 non-null   int64  
 11  TotalBsmtSF    1314 non-null   int64  
 12  1stFlrSF       1314 non-null   int64  
 13  2ndFlrSF       1314 non-null   int64  
 14  LowQualFinSF   1314 non-null   int64  
 15  GrLivArea      1314 non-null   int64  
 16  BsmtFullBath   1314 non-null   int64  
 17  BsmtHalfBath   1314 non-null   int64  
 18  FullBath       1314 no

In [79]:
medians = train.median(numeric_only=True)

train = train.fillna(medians)
val = val.fillna(medians)

Все, теперь, когда мы дропнули все строковые признаки и заполнили медианным значением численные, мы можем обучить модель и сделать предсказание

In [80]:
X_train = train.drop(['SalePrice'], axis=1)
y_train = train['SalePrice']

X_mean = X_train.mean()
X_std = X_train.std()
X_train = (X_train - X_mean) / X_std

linear_reg = LinearRegression(X_train.shape[1])
linear_reg.fit(X_train, y_train, learning_rate=1e-3, epochs=1000)

Epoch: 0
loss: 38644924416.0
Epoch: 100
loss: 23569612800.0
Epoch: 200
loss: 16029825024.0
Epoch: 300
loss: 11130231808.0
Epoch: 400
loss: 7855560192.0
Epoch: 500
loss: 5660424192.0
Epoch: 600
loss: 4187882496.0
Epoch: 700
loss: 3199485952.0
Epoch: 800
loss: 2535580416.0
Epoch: 900
loss: 2089243008.0


Хоть ошибка и выглядит огромной, но не стоит забывать, что это квадрат, то есть 42000 разброс для значений, которые начинаются от 100000, что вполне приемлимо

In [81]:
np.sqrt(1813816192.0)

np.float64(42588.92100065462)

Теперь предскажем значения на тестовых данных и посмотрим результат

In [82]:
X_val = val.drop(['SalePrice'], axis=1)
y_val = val['SalePrice']


X_val_mean = X_val.mean()
X_val_std = X_val.std()
X_val = (X_val - X_mean) / X_std


preds = linear_reg.predict(X_val)

In [83]:
preds

tensor([122330.5391, 282811.2812,  82929.4141, 152687.0938, 264370.2500,
         26838.7031, 199076.5000, 134016.4844,  26695.0312, 118595.7031,
        115663.7188,  86597.8594,  84607.4297, 184213.0000, 172165.3750,
        110923.1172, 183900.4688, 108354.8594,  86166.0625, 204706.1406,
        163522.6875, 181411.8281, 163482.3438, 101838.7266, 185035.9375,
        132807.0625, 170814.3281,  60875.5625, 164574.5000, 167498.0000,
        101100.4453, 249612.6562, 207147.1562,  61542.4453, 246333.6250,
        135834.5625, 121044.4688, 191082.4062, 276881.1250,  51182.0391,
        110497.5625, 219161.7500,  79942.3828, 277918.7500, 104383.9688,
        107844.7188,  75478.6875,  98757.0859, 330516.7500, 104746.7188,
         77666.5234, 199525.3750,  78717.4844, 277641.7500, 136638.2188,
        220070.9062, 198712.1562, 128140.4844, 134730.2500,  81426.6797,
         25392.3047, 121435.0156, 263149.3750, 228361.5625, 242847.5469,
        195813.0000,  62708.2500, 282820.3438,  767

In [84]:
def MSE(true_data, preds):
    true_data = torch.tensor(true_data.values, dtype=torch.float32)
    return torch.mean((true_data - preds)**2)

np.sqrt(MSE(y_val, preds))

C:\Users\jopajack\AppData\Local\Temp\ipykernel_9576\1038547749.py:5: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  np.sqrt(MSE(y_val, preds))


tensor(50444.8750)

Ну кстати, 50000 не так уж и плохо, я думал ошибка улетит сильно дальше, учитвыая что мы взяли базовую реализацию линейной регрессии для датасета, которому реально нужен фич инженеринг, и попросту бахнули на нем нейронку, всего лишь нормализовав значения фич.